# Clase 08 - Reto Final: Moneyball en la NHL

## Objetivo
Aplicar los conceptos de las 8 clases del diplomado para evaluar a la plantilla de los **New York Rangers** usando métricas avanzadas (Expected Goals, Corsi, Bloqueos). Tras identificar a los jugadores con peor rendimiento, propondremos **3 fichajes estrella** que ocupen esas posiciones exactas, maximizando el valor usando un presupuesto máximo de **$15 millones de dólares**.

## Dataset
Utilizaremos el dataset de Kaggle: `camnugent/predict-nhl-player-salaries`.

In [ ]:
# !uv run kaggle datasets download -d camnugent/predict-nhl-player-salaries --unzip -p data/nhl_salaries
from google.colab import files

print("Por favor, sube tu archivo kaggle.json:")
uploaded = files.upload()

for fn in uploaded.keys():
  print('Usuario subió el archivo "{name}" con longitud {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
# Mover credenciales a la carpeta correcta
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Descargar el dataset (Baseball Databank)
# Puedes cambiar 'open-source-sports/baseball-databank' por otro dataset de Kaggle si lo deseas
print("Descargando dataset...")
!kaggle datasets download -d camnugent/predict-nhl-player-salaries --unzip -p data
!ls data/

Dataset URL: https://www.kaggle.com/datasets/camnugent/predict-nhl-player-salaries
License(s): CC0-1.0
100%|████████████████████████████████████████| 183k/183k [00:00<00:00, 1.50MB/s]
100%|████████████████████████████████████████| 183k/183k [00:00<00:00, 1.47MB/s]


### 1. Cargar Datos y Análisis de Rendimiento Individual (NY Rangers)
Empezaremos evaluando el desempeño ofensivo y defensivo de la plantilla actual de los Rangers.

In [ ]:
import pandas as pd

df = pd.read_csv('data/train.csv', encoding='iso-8859-1')
df['Salary'] = pd.to_numeric(df['Salary'], errors='coerce')

nyr = df[df['Team'] == 'NYR'].copy()
nyr['CF%'] = (nyr['CF'] / (nyr['CF'] + nyr['CA'])) * 100
nyr['Net_TKA'] = nyr['TKA'] - nyr['GVA']
nyr['xG_Diff'] = nyr['G'] - nyr['ixG'] # Goles sobre lo esperado

print("--- ANÁLISIS OFENSIVO: GOLES vs EXPECTED GOALS (ixG) ---")
print("Jugadores rindiendo POR DEBAJO de sus goles esperados (mala definición/mala suerte):")
underperformers = nyr.sort_values('xG_Diff').head(3)
print(underperformers[['Last Name', 'Position', 'G', 'ixG', 'xG_Diff']])

print("\n--- ANÁLISIS DEFENSIVO: BLOQUEOS Y PÉRDIDAS ---")
print("Jugadores con más pérdidas netas (Takeaways vs Giveaways) y bajo CF%:")
defensive_liabilities = nyr[(nyr['Net_TKA'] < 0) & (nyr['CF%'] < 50)].sort_values('Net_TKA').head(3)
print(defensive_liabilities[['Last Name', 'Position', 'CF%', 'Net_TKA', 'iBLK']])

--- ANÁLISIS OFENSIVO: GOLES vs EXPECTED GOALS (ixG) ---
Jugadores rindiendo POR DEBAJO de sus goles esperados (mala definición/mala suerte):
    Last Name Position  G  ixG  xG_Diff
238  McDonagh        D  6  9.9     -3.9
481      Fast    RW/LW  6  7.6     -1.6
570     Skjei        D  5  5.9     -0.9

--- ANÁLISIS DEFENSIVO: BLOQUEOS Y PÉRDIDAS ---
Jugadores con más pérdidas netas (Takeaways vs Giveaways) y bajo CF%:
    Last Name Position        CF%  Net_TKA  iBLK
558    Holden        D  45.800748    -82.0    88
156      Nash    RW/LW  48.388626    -71.0    35
238  McDonagh        D  49.519890    -68.0   160


### 2. Identificación de Posiciones a Reforzar
Tomando a los jugadores menos eficientes de nuestros análisis anteriores, definiremos qué posiciones necesitamos buscar en el mercado.

In [3]:
# Extraer las posiciones de los peores evaluados
positions_to_replace = list(set(underperformers['Position'].tolist() + defensive_liabilities['Position'].tolist()))
print("Posiciones prioritarias a reforzar basándonos en bajo rendimiento:", positions_to_replace)

Posiciones prioritarias a reforzar basándonos en bajo rendimiento: ['RW/LW', 'D']


### 3. Búsqueda de Fichajes Estrella ("Moneyball") Mapeados a las Necesidades
Sabiendo qué perfiles tienen problemas buscaremos tres fichajes (cubriendo idealmente las posiciones necesarias) que superen el 50% de CF% y tengan alta producción ofensiva (PTS > 40), maximizando el *Value_Per_Million* (Puntos multiplicados por 1.5 + CF% por cada millón de salario).

In [4]:
targets = df[(df['Team'] != 'NYR') & (df['Salary'] > 0)].copy()
targets['CF%'] = (targets['CF'] / (targets['CF'] + targets['CA'])) * 100
targets['Value_Score'] = (targets['PTS'] * 1.5) + targets['CF%']
targets['Value_Per_Million'] = targets['Value_Score'] / (targets['Salary'] / 1000000)

# Filtrar solo jugadores de posiciones que necesitamos reemplazar
targets = targets[targets['Position'].isin(positions_to_replace)]

# Filtrar base de "Estrellas" (PTS > 40, CF% > 50)
stars = targets[(targets['PTS'] > 40) & (targets['CF%'] > 50)].copy()
best_stars = stars.sort_values(by='Value_Per_Million', ascending=False)

print("Top 10 Jugadores Disponibles en esas posiciones por Valor/Millón:")
print(best_stars[['First Name', 'Last Name', 'Position', 'Team', 'Salary', 'PTS', 'CF%', 'Value_Per_Million']].head(10))

Top 10 Jugadores Disponibles en esas posiciones por Valor/Millón:
    First Name    Last Name Position Team   Salary  PTS        CF%  \
25       David     Pastrnak    RW/LW  BOS   925000   70  62.932639   
249  Sebastian          Aho    RW/LW  CAR   925000   49  58.634701   
224       Zach     Werenski        D  CBJ   925000   47  56.918973   
592      Jakob  Silfverberg    RW/LW  ANA  3000000   49  51.610594   
286     Victor       Hedman        D  T.B  4250000   72  54.828571   
451      Wayne     Simmonds    RW/LW  PHI  4300000   54  57.801642   
36       Roman         Josi        D  NSH  4250000   49  53.667622   
339       Jake     Gardiner        D  TOR  4050000   43  56.233804   
62        Nick        Leddy        D  NYI  4500000   46  51.711457   
77       James         Neal    RW/LW  NSH  5000000   41  58.148631   

     Value_Per_Million  
25          181.548799  
249         142.848326  
224         137.750241  
592          41.703531  
286          38.312605  
451          

### 4. Selección de los 3 Fichajes Estrella bajo Presupuesto ($15M)

In [5]:
picked = []
total_salary = 0
budget = 15000000

# Seleccionamos los mejores asegurando que no nos pasemos del presupuesto
for idx, row in best_stars.iterrows():
    if len(picked) < 3 and (total_salary + row['Salary']) <= budget:
        picked.append(row)
        total_salary += row['Salary']

print("=== LOS 3 FICHAJES ESTRELLA IDEALES PARA CUBRIR NUESTRAS BAJAS ===")
for p in picked:
    print(f"Jugador: {p['First Name']} {p['Last Name']} | Pos: {p['Position']} | Equipo Actual: {p['Team']}")
    print(f"   Salario: ${p['Salary']:,.0f} | Puntos: {p['PTS']} | CF%: {p['CF%']:.2f}%\n")

print(f"💰 Salario Total Consumido: ${total_salary:,.0f} / ${budget:,.0f}")

=== LOS 3 FICHAJES ESTRELLA IDEALES PARA CUBRIR NUESTRAS BAJAS ===
Jugador: David Pastrnak | Pos: RW/LW | Equipo Actual: BOS
   Salario: $925,000 | Puntos: 70 | CF%: 62.93%

Jugador: Sebastian Aho | Pos: RW/LW | Equipo Actual: CAR
   Salario: $925,000 | Puntos: 49 | CF%: 58.63%

Jugador: Zach Werenski | Pos: D | Equipo Actual: CBJ
   Salario: $925,000 | Puntos: 47 | CF%: 56.92%

💰 Salario Total Consumido: $2,775,000 / $15,000,000
